# FMD bootstrap

Het enige bestand dat je met de hand in Fabric hoeft te zetten.

Vul hieronder in waar jouw fork staat, en draai het notebook. Het haalt
`NB_SETUP_FMD` en `NB_SETUP_BUSINESS_DOMAINS` uit die repo, stempelt dezelfde
drie waarden in hun bootstrap-cel, en zet ze in deze workspace.

Alle overige configuratie staat in `manifest.yaml` in de root van je fork; de
setup-notebooks lezen die zelf. Zie `manifest.example.yaml` voor de structuur.


In [ ]:
# =============================================================================
# Waar staat jouw fork?
# =============================================================================
repo_owner    = "bitmetric-fabric"   # GitHub-organisatie of -gebruiker
repo_name     = "FMD_FRAMEWORK"      # Naam van de repository
branch        = "main"               # Branch waaruit gedeployed wordt
folder_prefix = ""                   # Alleen invullen als src/ en config/ in een submap staan
github_token_key_vault = ""          # Key Vault-naam met een GitHub PAT; leeg = publieke repo
github_token_secret    = ""          # Secretnaam van de PAT in die Key Vault

# Deze waarden moeten gelijk zijn aan repository.* in manifest.yaml; de
# setup-notebooks controleren dat en stoppen bij een verschil.


## Setup-notebooks ophalen en plaatsen

In [ ]:
import base64
import json

import requests

SETUP_NOTEBOOKS = {
    "NB_SETUP_FMD": "fmd-manifest-bootstrap",
    "NB_SETUP_BUSINESS_DOMAINS": "bd-manifest-bootstrap",
}

FABRIC_API = "https://api.fabric.microsoft.com/v1"

workspace_id = notebookutils.runtime.context.get("currentWorkspaceId")
headers = {
    "Authorization": f"Bearer {notebookutils.credentials.getToken('pbi')}",
    "Content-Type": "application/json",
}

github_token = (
    notebookutils.credentials.getSecret(
        f"https://{github_token_key_vault}.vault.azure.net/", github_token_secret
    )
    if github_token_key_vault and github_token_secret
    else None
)
github_headers = {"Authorization": f"Bearer {github_token}"} if github_token else {}


def fetch(notebook_name):
    """Haalt het setup-notebook op uit de opgegeven fork."""
    prefix = f"{folder_prefix}/" if folder_prefix else ""
    url = (
        f"https://raw.githubusercontent.com/{repo_owner}/{repo_name}/{branch}/"
        f"{prefix}setup/{notebook_name}.ipynb"
    )
    response = requests.get(url, headers=github_headers)
    if response.status_code == 404:
        raise RuntimeError(f"Niet gevonden: {url}")
    response.raise_for_status()
    return json.loads(response.text), url


def stamp(notebook, cell_id):
    """Zet repo_owner/repo_name/branch/folder_prefix in de bootstrap-cel."""
    values = {
        "repo_owner": repo_owner,
        "repo_name": repo_name,
        "branch": branch,
        "folder_prefix": folder_prefix,
        "github_token_key_vault": github_token_key_vault,
        "github_token_secret": github_token_secret,
    }
    for cell in notebook["cells"]:
        if cell.get("id") != cell_id:
            continue
        source = cell["source"]
        source = source if isinstance(source, list) else source.splitlines(keepends=True)
        for index, line in enumerate(source):
            key = line.split("=", 1)[0].strip()
            if key in values:
                ending = "\n" if line.endswith("\n") else ""
                source[index] = f'{key:<13} = "{values[key]}"{ending}'
                del values[key]
        cell["source"] = source
        if values:
            raise RuntimeError(f"Niet gestempeld in {cell_id}: {sorted(values)}")
        return notebook
    raise RuntimeError(
        f"Cel '{cell_id}' ontbreekt. Draait deze fork nog op een versie van voor "
        f"het manifest (PR 3)?"
    )


def find_item(display_name):
    response = requests.get(f"{FABRIC_API}/workspaces/{workspace_id}/items?type=Notebook", headers=headers)
    response.raise_for_status()
    for item in response.json().get("value", []):
        if item["displayName"] == display_name:
            return item["id"]
    return None


def publish(display_name, notebook):
    payload = base64.b64encode(json.dumps(notebook).encode("utf-8")).decode("utf-8")
    definition = {
        "format": "ipynb",
        "parts": [
            {"path": "notebook-content.ipynb", "payload": payload, "payloadType": "InlineBase64"}
        ],
    }
    item_id = find_item(display_name)
    if item_id:
        url = f"{FABRIC_API}/workspaces/{workspace_id}/items/{item_id}/updateDefinition"
        body = {"definition": definition}
        verb = "bijgewerkt"
    else:
        url = f"{FABRIC_API}/workspaces/{workspace_id}/items"
        body = {"displayName": display_name, "type": "Notebook", "definition": definition}
        verb = "aangemaakt"

    response = requests.post(url, headers=headers, json=body)
    if response.status_code not in (200, 201, 202):
        raise RuntimeError(f"{display_name}: HTTP {response.status_code} {response.text}")
    print(f"{display_name} {verb}")


for notebook_name, cell_id in SETUP_NOTEBOOKS.items():
    notebook, source_url = fetch(notebook_name)
    publish(notebook_name, stamp(notebook, cell_id))
    print(f"   bron: {source_url}")

print(
    "\nKlaar. Open NB_SETUP_FMD en draai hem. Zorg dat manifest.yaml in de root "
    f"van {repo_owner}/{repo_name} (branch '{branch}') staat en ingevuld is."
)
